## 1 - Project Setup
This section initializes the complete workspace environment, mounts Google Drive and extracts the dataset from the .zip archive directly to the local instance storage (/content/dataset).


In [ ]:
%pip install -q timm ipywidgets

import os
import glob
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm import tqdm
import warnings
import random
import gc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Environment setup: works both on Colab (Drive-mounted) and on a local VM/JupyterLab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = '/content/drive/MyDrive/internship-deepfake-forensic'
    DATASET_DIR = '/content/dataset'
    MODELS_DIR = os.path.join(DRIVE_ROOT, 'deepfake_models_final')

    if not os.path.exists(DATASET_DIR):
        print("Extracting the dataset... (this might take a few minutes)")
        !unzip -q {os.path.join(DRIVE_ROOT, 'deepfake_dataset.zip')} -d {DATASET_DIR}
        print("Extraction completed!")
    else:
        print("Dataset already present and ready to use!")
else:
    DATA_ROOT = '/home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/02_Extended_Framework'
    DATASET_DIR = os.path.join(DATA_ROOT, 'Datasets')
    MODELS_DIR = os.path.join(DATA_ROOT, 'deepfake_models_final')
    print(f"Running outside Colab. Using local paths under: {DATA_ROOT}")

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Dataset directory: {DATASET_DIR}")
print(f"Models directory:  {MODELS_DIR}")

## 2 - Sanity and Visual Check
This section handles the path correction and validation of the dataset. Since the original CSV may contain paths from different environments we dynamically update them to match the Colab local storage. After remapping, the script performs a Sanity Check and a random selection of images is visualized to ensure data integrity and correct labeling.

In [ ]:
csv_path = os.path.join(DATASET_DIR, 'master_dataset.csv')
df = pd.read_csv(csv_path)

df['path'] = df['path'].apply(lambda p: os.path.join(DATASET_DIR, str(p)))

print("--- EXAMPLES OF PATHS ---")
for p in df['path'].sample(15):
    status = "OK" if os.path.exists(p) else "ERROR"
    print(f"{status} | Path: {p}")

In [ ]:
def preview_random_images(df_list, num_per_df=5):
    """
    Previews random images from a list of DataFrames.
    Displays Master dataset samples and Celeb-DF samples side by side.
    """
    # Combine a sample from each provided dataframe
    samples = []
    for df in df_list:
        samples.append(df.sample(min(num_per_df, len(df))))

    combined_sample = pd.concat(samples).sample(frac=1) # Shuffle the preview

    total_images = len(combined_sample)
    plt.figure(figsize=(20, 5))
    plt.suptitle("Random Preview of the Dataset", fontsize=16, fontweight='bold', y=1.05)

    for i, (idx, row) in enumerate(combined_sample.iterrows()):
        plt.subplot(1, total_images, i+1)

        try:
            img = Image.open(row['path'])
            plt.imshow(img)
        except Exception as e:
            plt.text(0.5, 0.5, f"Error\n{row['dataset']}", ha='center', va='center')

        label_text = "FAKE" if int(row['label']) == 1 else "REAL"
        color = 'red' if int(row['label']) == 1 else 'green'

        # Display Dataset, Split, and Label
        plt.title(f"{row['dataset']} ({row['split']})\n{label_text}", color=color, fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

preview_random_images([df], num_per_df=4)

## 3 - Dataset and Augmentation Baseline

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, real_dirs, fake_dirs, transform=None):
        self.filepaths, self.labels = [], []
        self.transform = transform
        extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
        for d in real_dirs:
            for ext in extensions:
                paths = sorted(glob.glob(os.path.join(d, ext)))
                self.filepaths.extend(paths)
                self.labels.extend([0] * len(paths))
        for d in fake_dirs:
            for ext in extensions:
                paths = sorted(glob.glob(os.path.join(d, ext)))
                self.filepaths.extend(paths)
                self.labels.extend([1] * len(paths))

    def __len__(self): return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor([self.labels[idx]], dtype=torch.float32)

class RandomJpegCompression:
    """
    Applies a single random JPEG compression.
    """
    def __init__(self, min_quality=50, max_quality=90, p=0.5):
        self.min_quality = min_quality
        self.max_quality = max_quality
        self.p = p

    def __call__(self, img):
        if torch.rand(1).item() > self.p:
            return img

        quality = torch.randint(self.min_quality, self.max_quality + 1, (1,)).item()
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

# BASELINE AUGMENTATION
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Fail-safe resize
    RandomJpegCompression(min_quality=60, max_quality=95, p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Fail-safe resize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 4 - Models Architecture & Models Weights Calculation (Parameters)

In [ ]:
class StandardEfficientNet(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.model = models.efficientnet_b0(weights=weights)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

class DeitTiny(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        self.model = timm.create_model('deit_tiny_patch16_224', pretrained=pretrained)
        n_features = self.model.head.in_features
        self.model.head = nn.Linear(n_features, num_classes)

    def forward(self, x):
        return self.model(x)

class HybridEfficientNet(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.backbone = models.efficientnet_b0(weights=weights).features
        in_channels = 1280
        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 1, bias=False),
            nn.BatchNorm2d(in_channels // 8), nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, in_channels, 1, bias=False), nn.Sigmoid()
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4), nn.Linear(in_channels, 256), nn.ReLU(inplace=True),
            nn.Dropout(p=0.4), nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        features = features * self.attention(features)
        return self.classifier(torch.flatten(self.pool(features), 1))

In [ ]:
model_std = StandardEfficientNet(pretrained=False)
model_deit = DeitTiny(pretrained=False)
model_hyb = HybridEfficientNet(pretrained=False)

def count_parameters(model):
    """Calculates the total number of trainable parameters of a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("--- MODELS WEIGHT COMPARISON (Trainable Parameters)--- ")

params_std  = count_parameters(model_std)
params_deit = count_parameters(model_deit)
params_hyb  = count_parameters(model_hyb)

print(f"Standard EfficientNet-B0: {params_std:>12,} parameters")
print(f"DeiT-Tiny:                {params_deit:>12,} parameters")
print(f"Hybrid EfficientNet-B0:   {params_hyb:>12,} parameters")

# These instances were only needed to count parameters, not for training
del model_std, model_deit, model_hyb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5 - Training Engine

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0.001, path='best_model.pth'):
        self.patience, self.delta, self.path = patience, delta, path
        self.counter, self.best_score, self.early_stop = 0, None, False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    #ModelCheckPoint - save the model when improve
    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

def train_and_validate(model, train_loader, val_loader, epochs, optimizer, criterion, device, early_stopper, scheduler=None):
    scaler = torch.amp.GradScaler('cuda')

    history = {
        'epoch': [],
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    for epoch in range(epochs):
        # TRAINING PHASE
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="batch", leave=False)

        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * images.size(0)

            predicted = (outputs > 0.0).float()
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            train_bar.set_postfix(loss=loss.item(), acc=train_correct/train_total)

        # VALIDATION PHASE
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", unit="batch", leave=False)

        with torch.no_grad():
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)

                predicted = (outputs > 0.0).float()
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        avg_train_loss = train_loss / train_total
        avg_val_loss = val_loss / val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total

        print(f"END EPOCH {epoch+1} -> Train Loss: {avg_train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.4f}")

        history['epoch'].append(epoch + 1)
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        # SCHEDULER LOGIC
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(avg_val_loss)
            else:
                scheduler.step()

        early_stopper(avg_val_loss, model)
        if early_stopper.early_stop:
            print(">> Early stopping activate!")
            break
        print("-" * 30)

    history_path = early_stopper.path.replace('.pth', '_history.csv')
    pd.DataFrame(history).to_csv(history_path, index=False)
    print(f"Training history saved successfully in: {history_path}")

    return history

def load_or_train(model, model_path, train_loader, val_loader, epochs, optimizer, criterion, device, early_stopper, scheduler=None):
    history = None
    if os.path.exists(model_path):
        print(f"Model found! Skipping training and loading weights from:\n{model_path}")
        model.load_state_dict(torch.load(model_path, map_location=device))

        history_path = model_path.replace('.pth', '_history.csv')
        if os.path.exists(history_path):
            history = pd.read_csv(history_path).to_dict(orient='list')
            print(f"Training history loaded from: {history_path}")
    else:
        print(f"Model not found. Starting the training procedure...")
        history = train_and_validate(
            model, train_loader, val_loader, epochs, optimizer,
            criterion, device, early_stopper, scheduler
        )
        # Load the best model saved by EarlyStopping at the end of training
        model.load_state_dict(torch.load(model_path, map_location=device))

    return history

## 6 - Data Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

path_intra = os.path.join(MODELS_DIR, 'baseline', 'path')
os.makedirs(path_intra, exist_ok=True)
print(f"In this first step the models are saved in: {path_intra}")

BATCH_SIZE = 32
MAX_EPOCHS = 15
NUM_WORKERS = 4

# SETUP
tr_reals = [f'{DATASET_DIR}/FEI_MORPHV2_DATASET/train/original/', f'{DATASET_DIR}/CELEBDFV3_DATASET/train/original/']
tr_fakes = [f'{DATASET_DIR}/FEI_MORPHV2_DATASET/train/fake/', f'{DATASET_DIR}/CELEBDFV3_DATASET/train/fake/']
vl_reals = [f'{DATASET_DIR}/FEI_MORPHV2_DATASET/val/original/', f'{DATASET_DIR}/CELEBDFV3_DATASET/val/original/']
vl_fakes = [f'{DATASET_DIR}/FEI_MORPHV2_DATASET/val/fake/', f'{DATASET_DIR}/CELEBDFV3_DATASET/val/fake/']

train_ds = DeepfakeDataset(tr_reals, tr_fakes, train_transforms)
val_ds = DeepfakeDataset(vl_reals, vl_fakes, val_test_transforms)

def get_criterion_from_labels(labels):
    num_reals = labels.count(0)
    num_fakes = labels.count(1)
    balance_ratio = num_reals / num_fakes if num_fakes > 0 else 1.0
    print(f"Class balance computed: {num_reals} Real / {num_fakes} Fake (pos_weight: {balance_ratio:.4f})")
    return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([balance_ratio]).to(device))

print("\nComputing class weights:")
criterion = get_criterion_from_labels(train_ds.labels)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS, persistent_workers=True)

## 7 - Training Phase

### 7.1 - EfficientNet-B0

In [ ]:
#EFFICIENTNET STANDARD
model_std_b0 = StandardEfficientNet().to(device)
optimizer_b0 = optim.AdamW(model_std_b0.parameters(), lr=1e-4, weight_decay=1e-5) #AdamW to optimizer
scheduler_std_b0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b0, mode='min', factor=0.5, patience=2
) #ReduceLROnPlateau - to recude LR if the val loss do not improve
path_std_b0 = os.path.join(path_intra, "std_b0_step1.pth")
early_stopper_std_b0 = EarlyStopping(patience=10, path=path_std_b0)

print("--- STANDARD EFFICIENTNET-B0 --> START TRAINING ---")
load_or_train(
    model_std_b0,
    path_std_b0,
    train_loader,
    val_loader,
    MAX_EPOCHS,
    optimizer_b0,
    criterion,
    device,
    early_stopper_std_b0,
    scheduler_std_b0
)

# Free GPU memory before training the next model
del model_std_b0, optimizer_b0, scheduler_std_b0
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 7.2 - Deit-Tiny

In [ ]:
#DEIT
warnings.filterwarnings("ignore", module="huggingface_hub")

model_deit = DeitTiny().to(device)
optimizer_deit = optim.AdamW(model_deit.parameters(), lr=5e-5, weight_decay=1e-5)
scheduler_deit = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_deit, mode='min', factor=0.5, patience=2
)
path_deit = os.path.join(path_intra, "deit_step1.pth")
early_stopper_deit = EarlyStopping(patience=10, path=path_deit)

print("--- DEIT-TINY --> START TRAINING ---")
load_or_train(
    model_deit,
    path_deit,
    train_loader,
    val_loader,
    MAX_EPOCHS,
    optimizer_deit,
    criterion,
    device,
    early_stopper_deit,
    scheduler_deit
)

# Free GPU memory before training the next model
del model_deit, optimizer_deit, scheduler_deit
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 7.3 - Hybrid EfficientNet-B0

In [ ]:
#HYBRID EFFICIENTNET
model_hyb_b0 = HybridEfficientNet().to(device)
optimizer_hyb_b0 = optim.AdamW(model_hyb_b0.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler_hyb_b0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_hyb_b0,
    mode='min',
    factor=0.5,
    patience=2
)
path_hyb_b0 = os.path.join(path_intra, "hyb_b0_step1.pth")
early_stopper_hyb_b0 = EarlyStopping(patience=10, path=path_hyb_b0)

print("--- HYBRID EFFICIENTNET-B0 --> START TRAINING ---")
load_or_train(
    model_hyb_b0,
    path_hyb_b0,
    train_loader,
    val_loader,
    MAX_EPOCHS,
    optimizer_hyb_b0,
    criterion,
    device,
    early_stopper_hyb_b0,
    scheduler_hyb_b0)

# Free GPU memory
del model_hyb_b0, optimizer_hyb_b0, scheduler_hyb_b0
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8 - Learning Curves Evaluation
This section generates and saves the training and validation plots (Loss and Accuracy) for the evaluated models.

In [ ]:
def plot_learning_curves(csv_path, model_name="Model", curves_path=None):
    """
    Reads the history CSV and saves an image with the Loss and Accuracy curves
    to the specified curves_path.
    """
    if curves_path is None:
        curves_path = os.path.join(MODELS_DIR, 'baseline', 'curves')

    if not os.path.exists(csv_path):
        print(f"Error: The file {csv_path} does not exist.")
        return

    # Load the saved data
    df = pd.read_csv(csv_path)
    epochs = df['epoch']

    plt.figure(figsize=(14, 5))

    # PLOT 1: LOSS
    plt.subplot(1, 2, 1)
    plt.plot(epochs, df['train_loss'], label='Train Loss', color='blue', linewidth=2)
    plt.plot(epochs, df['val_loss'], label='Validation Loss', color='orange', linewidth=2, linestyle='--')
    plt.title(f'Loss Trend - {model_name}', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.xticks(epochs)

    # PLOT 2: ACCURACY
    plt.subplot(1, 2, 2)
    plt.plot(epochs, df['train_acc'], label='Train Accuracy', color='green', linewidth=2)
    plt.plot(epochs, df['val_acc'], label='Validation Accuracy', color='red', linewidth=2, linestyle='--')
    plt.title(f'Accuracy Trend - {model_name}', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.xticks(epochs)

    plt.tight_layout()

    # Ensure the target directory exists
    os.makedirs(curves_path, exist_ok=True)

    # Generate the file name and final output path
    file_name = f"curves_{model_name.replace(' ', '_').lower()}.png"
    output_path = os.path.join(curves_path, file_name)

    # Save and display the plot
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Plot successfully saved as: {output_path}")

plot_learning_curves(os.path.join(path_intra, 'std_b0_step1_history.csv'), model_name='Std Efficient-Net')
plot_learning_curves(os.path.join(path_intra, 'deit_step1_history.csv'), model_name='Deit-Tiny')
plot_learning_curves(os.path.join(path_intra, 'hyb_b0_step1_history.csv'), model_name='Hyb Efficient-Net')
